In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import copy
import plotly.graph_objects as go
from PIL import Image


In [5]:
def load_samples(folder, id = 0):
    inp = np.load(f"{folder}/sample_{id}_inp.npy")
    out = np.load(f"{folder}/sample_{id}_out.npy")
    pred = np.load(f"{folder}/sample_{id}_pred.npy")
    return inp, out, pred

def plot_samples(inp, out, pred, pred_sc = None, dim = 4, folder = None):
    
    images = [inp, out, pred]
    if pred_sc is not None:
        images.append(pred_sc)
    row_labels = ['Input', 'Ground Truth', 'Prediction FM']
    if pred_sc is not None:
        row_labels.append("Prediction Scr.")
    
    if dim == 6:
        col_labels = ["$ρ$", "$u$", "$v$", "$p$", "$b_x$", "$b_y$"]
    elif dim == 5:
        col_labels = ["$ρ$", "$u$", "$v$", "$w$", "$p$"]
    
    fig, axes = plt.subplots(len(images), dim, figsize=(3 * dim, 3 * len(images))) 

    for j in range(dim):
        for i in range(len(images)):
            if i > 0:
                vmin =  np.min(out[j])
                vmax =  np.max(out[j])
            else:
                vmin =  np.min(inp[j])
                vmax =  np.max(inp[j])
            axes[i,j].imshow(images[i][j],cmap='gist_ncar', vmin = vmin, vmax = vmax)
            axes[i,j].axis('off')
    
    # Add column labels
    for ax, col_label in zip(axes[0], col_labels[:dim]):
        ax.set_title(col_label, fontsize=25)

     # Row labels (outside the first column of images)
    for i, label in enumerate(row_labels):
        axes[i, 0].annotate(label, xy=(0, 0.5), xytext=(-axes[i, 0].bbox.width * 0.1, 0),
                            xycoords='axes fraction', textcoords='offset points',
                            ha='right', va='center', rotation=90, fontsize=25)

    plt.tight_layout()
    
    if folder is not None:
        plt.savefig(f"{folder}.pdf", dpi = 400)
        plt.savefig(f"{folder}.png", dpi = 400)
    plt.show()

In [6]:
# Function to build bounding box
def create_bbox(x0=0, y0=0, z0=0, size=64, color='black'):
    # 8 corners
    c = np.array([
        [0, 0, 0],
        [size-1, 0, 0],
        [size-1, size-1, 0],
        [0, size-1, 0],
        [0, 0, size-1],
        [size-1, 0, size-1],
        [size-1, size-1, size-1],
        [0, size-1, size-1]
    ]) + np.array([x0, y0, z0])

    # Define connections for cube edges
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),  # bottom square
        (4, 5), (5, 6), (6, 7), (7, 4),  # top square
        (0, 4), (1, 5), (2, 6), (3, 7)   # vertical edges
    ]

    # Create line segments
    xb, yb, zb = [], [], []
    for i, j in edges:
        xb += [c[i, 0], c[j, 0], None]
        yb += [c[i, 1], c[j, 1], None]
        zb += [c[i, 2], c[j, 2], None]

    return go.Scatter3d(
        x=xb, y=yb, z=zb,
        mode='lines',
        line=dict(color=color, width=2),
        showlegend=False
    )

In [7]:
id = 123
N = 128
which_data = "ns_shear3d_mm"

if "ellipse" in which_data:
    folder = f"/cluster/work/math/braonic/TrainedModels/OOD_Generalization/pdegym_plus3d/PDEGYM_PLUS_3d_10ep_ViTB_regression/finetuned/{which_data}/post_pdegym_plus3d_FT_{which_data}_non_native_{N}/predictions_{which_data}_id14"
elif "kh3d" in which_data:
    folder = f"/cluster/work/math/braonic/TrainedModels/OOD_Generalization/pdegym_plus3d/PDEGYM_PLUS_3d_10ep_ViTB_regression/finetuned/{which_data}/post_pdegym_plus3d_FT_{which_data}_non_native_{N}/predictions_{which_data}_id14"
elif "curved" in which_data:
    #folder = f"/cluster/work/math/braonic/TrainedModels/OOD_Generalization/pdegym_plus/PDEGYM_PLUS_10ep_ViTB_regression/finetuned/{which_data}/trial_FT_{which_data}_non_native_{N}/predictions_{which_data}_id14"
    folder = "/cluster/work/math/braonic/TrainedModels/OOD_Generalization/pdegym_plus/PDEGYM_PLUS_10ep_ViTB_regression/finetuned/eul3d_mix1/post_training4700_vol2_4090_8gpus_bs3_acc4_FT_eul3d_mix1_non_native_4700/predictions_eul_riemann_curved3d_id14"
elif "shear3d" in which_data:
    folder = "/cluster/work/math/braonic/TrainedModels/OOD_Generalization/eul_ns3d_mix1/TURBO_MASK_scratch_Base_10ep_8gpus_bs3_4acc_10000/predictions_ns_shear3d_id14"
elif "shear3d_mm" in which_data:
    folder = "/cluster/work/math/braonic/TrainedModels/OOD_Generalization/eul_ns3d_mix1/TURBO_MASK_scratch_Base_10ep_8gpus_bs3_4acc_10000/predictions_ns_shear3d_mm_test"
inp, out_, pred1_ = load_samples(folder, id)
#folder = f"/cluster/work/math/braonic/TrainedModels/OOD_Generalization/{which_data}/ViT_SMALL_scratch_33M/scaling_{N}/predictions_{which_data}_id14"
#_, _, pred2_ = load_samples(folder, id)

pred2_ = pred1_

In [8]:
import kaleido
kaleido.get_chrome_sync()


PosixPath('/cluster/home/braonic/.local/lib/python3.11/site-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

In [10]:
D = 1
which = "ft"
offset = 90

if which_data == "eul_riemann_kh3d" or which_data == "eul_riemann_curved3d":
    isomin = 0.05
    isomax = 0.70
    opacity = 1.0
    
    if "curved" in which_data and D == 1:
        isomin = 0.05
        isomax = 0.80

    pred1 = pred1_
    pred2 = pred2_
    out = out_

    surface_count = 6
    scale = 1
elif "shear3d" in which_data:

    isomin = 0.25
    isomax = 0.8
    opacity = 0.2
    
    pred1 = pred1_
    pred2 = pred2_
    out = out_

    surface_count = 15
    scale = 2

else:
    if D == 0:
        isomin = 0.22
        isomax = 0.6
        surface_count = 6
    elif D == 1:
        isomin = 0.22
        isomax = 0.66
        surface_count = 6
    elif D == 4:
        isomin = 0.22
        isomax = 0.6
        surface_count = 8
    
    scale = 3
    opacity = 0.175
    
    pred1 = pred1_[...,::-1,:,:]
    pred2 = pred2_[...,::-1,:,:]
    out = out_[...,::-1,:,:]

# Normalize 3 datasets
data2 = (pred1[D] - np.min(pred1[D])) / (np.max(pred1[D]) - np.min(pred1[D]))
data3 = (out[D] - np.min(pred1[D])) / (np.max(pred1[D]) - np.min(pred1[D]))
data1 = (pred2[D] - np.min(pred1[D])) / (np.max(pred1[D]) - np.min(pred1[D]))  # <- third dataset

if which == "ft":
    data1 = data2
elif which == "scratch":
    data1 = data1
elif which == "gt":
    data1 = data3
else:
    raise ValueError('which needs to be in the ones above') 

# Common coordinates
x = np.repeat(np.arange(64), 64*64)
y = np.tile(np.repeat(np.arange(64), 64), 64)
z = np.tile(np.arange(64), 64*64)

vol1 = go.Isosurface(x=x, y=y, z=z, value=data1.flatten(),
                     isomin=isomin, isomax=isomax, opacity=opacity, surface_count=surface_count,
                     colorscale='jet', showscale=False)

bbox1 = create_bbox(x0=0)

fig = go.Figure(data=[vol1,bbox1])# vol2, vol3, bbox1, bbox2, bbox3])

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False, range=[0, 63]),
        yaxis=dict(visible=False, range=[0, 63]),
        zaxis=dict(visible=False, range=[0, 63]),
        bgcolor='white'
    ),
    margin=dict(l=1, r=0, b=0, t=0),
    scene_camera=dict(eye=dict(x=1.55, y=1.75, z=1.5)),
    #scene_camera=dict(eye=dict(x=1.2, y=0.8, z=0.7)),
)

fig.show()
file = f"Figures/{which_data}/samples/sample_{id}_trainN_{N}_dimension_{D}_{which}.pdf"
fig.write_image(f"Figures/{which_data}/samples/4700sample_{id}_trainN_{N}_dimension_{D}_{which}.png", scale=scale)   
fig.write_image(f"Figures/{which_data}/samples/4700sample_{id}_trainN_{N}_dimension_{D}_{which}.pdf", scale=scale)   

<br>
<br>
<br>
<br>
<br>
<br>
<br>

In [5]:
def to_primitive(rho, mx, my, mz, E):
    vx = mx / rho
    vy = my / rho
    vz = mz / rho
    P = 0.4 * (E - 0.5 * (mx ** 2 + my ** 2 + mz ** 2) / rho)
    return rho, vx, vy, vz, P

In [3]:
import netCDF4 as nc

# Path to the NetCDF file
file_path = "/cluster/work/math/camlab-data/synthetic/IEU_3D_MacroMicroCylindricalShearFlow.nc"

# Open the dataset
ds = nc.Dataset(file_path, mode="r")

# Print all variables
print("Variables in the dataset:")
for var_name in ds.variables.keys():
    print(var_name)

# Optionally, inspect details of a variable
# print(ds.variables['temperature'])   # Example, if 'temperature' exists

# Close the dataset when done


#rho = ds["rho"][:100]
"""print("1")
u = ds["u"][:100]
print("1")
v = ds["v"][:100]
print("1")
w = ds["w"][:100]"""
'''
print("1")
E = ds["E"][:100]
print("1")

rho, vx, vy, vz, P = to_primitive(rho, mx, my, mz, E)
'''

print(ds['u'].shape)
ds.close()


Variables in the dataset:
macro
micro
time
x
y
z
u
v
w
(10, 1000, 5, 128, 128, 128)


In [16]:
import numpy as np
#means = [np.mean(rho), np.mean(vx), np.mean(vy), np.mean(vz), np.mean(P)]
#stds =  [np.std(rho), np.std(vx), np.std(vy), np.std(vz), np.std(P)]

means = [np.mean(u), np.mean(v), np.mean(w)]
stds =  [np.std(u), np.std(v), np.std(w)]

In [17]:
print(means)
print(stds)

[-1.6370905e-13, -1.7244019e-11, -3.4197e-12]
[0.29187253, 0.29185304, 0.19256265]


In [ ]:
[2.535, 5.356, 0.0, 0.0, 77.09]
[2.002, 5.654, 0.441, 0.305, 81.72]